# CS103 Elements of AI Final Project

Read the project description carefully and follow all instructions. This project is an individual task: you may discuss ideas with others, but you must not work on solutions together or share code. Any form of plagiarism or unauthorized collaboration will result in an F for the entire course.

## Guideline

Primary TA: "" ()

**How to submit**
* Follow only the 🛑 <mark> TODO</mark> instructions. **Do not** modify other code unless a TODO explicitly asks for it.
* You don't need to submit the original file. Your score will be automatically posted on the LeaderBoard
* Due date: 23:59:59 **_____** (__). Late submission is accepted until 23:59:59 **_____** (__) with a 20% penalty.

**Before submitting**
* Restart the runtime and run all cells in order.
* Complete every 🛑 <mark> TODO</mark>.

**Restrictions**
* Use only the packages and resources provided for this homework unless additional tools are explicitly allowed.
* The recipe database used in the ScienceWorld section should be treated as an external resource for retrieval, not as a solution file to inspect manually.

## Overview

In final project, you will build LangGraph-based agent to solve the CS103ScienceWorld tasks.

## Note

* You will need to use VectorDB. We will give you embedded vectors for the recipe corpus so you don't need to embed all texts.
* `BAAI/bge-m3` should be used as embedding model to encode query.
* The tasks of the final project is complete different from the original task. So you can't refer to the solution of the original tasks. Rather, we will give you `tiny` and `seen` tasks to practice them.

In [1]:
import os
import re
from typing import Any, Dict, List, Optional, Sequence, TypedDict

from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

from cs103_scienceworld import CS103ScienceWorldFinalProjectEnv

Here is the llm setting. You may attach tools with given `ChatOpenAI` object.

In [6]:
CHATOPENAI_KWARGS = {
    "model": "Qwen/Qwen3-Omni-30B-A3B-Instruct",
    # "model": "Qwen/Qwen3-30B-A3B-Thinking",
    "temperature": 0.7,
    "base_url": "http://192.249.19.233:7000/v1",
    "timeout": 10,
    "max_retries": 0,
    "api_key": "LOCAL_DUMMY"
}

llm = ChatOpenAI(
    **CHATOPENAI_KWARGS,
)

Setup your `Env` Object and look what tasks are available

In [3]:
env = CS103ScienceWorldFinalProjectEnv(envStepLimit=40)

task_names = env.get_task_names()

print(len(task_names))
print(task_names)

8
['cook-mini', 'cook-seen', 'corrosion-mini', 'corrosion-seen', 'recipe-pipeline-tiny', 'recipe-pipeline-seen', 'corrode-circuit-tiny', 'corrode-circuit-seen']


Here is the sandbox for the `CS103ScienceWorldFinalProjectEnv`(Same as assignment 7). The detailed explanation of the usage of this library is in Assignment 7. Play with the new tasks with tiny and seen tasks.

In [ ]:
env = CS103ScienceWorldFinalProjectEnv()

task_num = 0
var_num = 0
task_name = task_names[task_num]
env.load(task_name, var_num, simplificationStr="")

initial_observation, initial_dictionary = env.reset()
print(f"Starting {task_name}")
print(env.get_task_description())

userInputStr = "look around"
while (userInputStr != "exit"):
    if (userInputStr == "help"):
        print("Possible actions: ")
        for actionStr in env.get_possible_actions():
            print("\t" + str(actionStr))

    elif (userInputStr == "objects"):
        print("Possible objects (one referent listed per object): ")
        for actionStr in env.get_possible_objects():
            print("\t" + str(actionStr))

    elif (userInputStr == "valid"):
        print("Valid action-object combinations:")
        print(env.get_valid_action_object_combinations_with_templates())

    elif (userInputStr == 'goals'):
        print(env.get_goal_progress())

    else:
        # Send user input, get response
        observation, reward, isCompleted, info = env.step(userInputStr)
        score = info['score']
        print("\n" + observation)
        print("Reward: " + str(reward))
        print("Score: " + str(score))
        print("isCompleted: " + str(isCompleted))
        # print("info: " + str(info))

    print("'help' lists valid action templates, 'objects' lists valid objects, use <tab> to list valid actions. ")
    print("'goals' lists progress on subgoals.")
    print("type 'exit' to quit.")

    userInputStr = input('> ')

🛑 <mark> TODO</mark> <br>
You can freely define your `Node`, `Tool`, `Graph` object. Also you are allowed to use `Chroma` to solve the retrieval task. We will use your graph object directly for grading unseen tasks. For grading, you **MUST** use `StateGraph` type in `LangGraph`.

In [7]:
# Your Section #

class AgentState(TypedDict, total=False):
    observation: str
    info: Dict[str, Any]
    valid_actions: List[str]
    corpus: List[str]
    action: str


def extract_text_content(content: Any) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                parts.append(item.get("text", ""))
            else:
                parts.append(str(item))
        return "\n".join(part for part in parts if part)
    return str(content)


def parse_action(valid_actions: Sequence[str], raw_text: str) -> Optional[str]:
    if not raw_text:
        return None

    lookup = {action.lower(): action for action in valid_actions}
    text = raw_text.strip()

    match = re.search(r"Action\s*:\s*(.+)", text, flags=re.IGNORECASE)
    if match:
        candidate = match.group(1).strip().strip("`").strip()
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    stripped = text.strip("`").strip()
    if stripped.lower() in lookup:
        return lookup[stripped.lower()]

    for line in text.splitlines():
        candidate = line.strip().strip("`").strip()
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    for action in valid_actions:
        if action.lower() in text.lower():
            return action

    return None


def fallback_action(valid_actions: Sequence[str]) -> str:
    for action in valid_actions:
        if action.lower().startswith("wait"):
            return action
    return valid_actions[0]


llm = ChatOpenAI(
    **CHATOPENAI_KWARGS,
)


def decide_action(state: AgentState) -> AgentState:
    observation = state.get("observation", "")
    info = state.get("info", {})
    valid_actions = state.get("valid_actions", [])
    corpus = state.get("corpus", [])

    query = " ".join([str(info.get("taskDesc", "")), observation]).strip().lower()
    query_words = set(re.findall(r"[a-z0-9]+", query))
    ranked_docs = []
    for doc in corpus:
        doc_words = set(re.findall(r"[a-z0-9]+", doc.lower()))
        score = len(query_words & doc_words)
        if score > 0:
            ranked_docs.append((score, doc))
    ranked_docs.sort(key=lambda item: (-item[0], len(item[1])))
    docs = "\n\n".join(doc for _, doc in ranked_docs[:3]) or "<none>"
    candidates = "\n".join(f"- {action}" for action in valid_actions)

    prompt = f"""You are controlling a ScienceWorld agent.
Pick exactly one action from the valid action list.

Task: {info.get('taskDesc', '')}
Observation:
{observation}

Info:
{info}

Corpus snippets:
{docs}

Valid actions:
{candidates}

Reply like this:
Thought: short reasoning
Action: exact action"""

    response = llm.invoke(prompt)
    raw_output = extract_text_content(response.content)
    action = parse_action(valid_actions, raw_output)
    if action is None:
        action = fallback_action(valid_actions)
    return {"action": action}


state_graph = StateGraph(AgentState)
state_graph.add_node("decide_action", decide_action)
state_graph.add_edge(START, "decide_action")
state_graph.add_edge("decide_action", END)




## Important
The cell below is for grading. You can pass your graph object and student id for grading. The grading is conducted using 3 random variations per each task. Your score might be different per each attempt.

* Note that this cell may takes some times

In [8]:
report = env.grade_state_graph(
    state_graph=state_graph,
    student_id="20260000",
    print_progress=True,
)

print(report)
print(f"score: {report.total_score}/{report.max_score}")

Grading progress: 1/10 episodes completed
Grading progress: 2/10 episodes completed
Grading progress: 3/10 episodes completed
Grading progress: 4/10 episodes completed
Grading progress: 5/10 episodes completed
Grading progress: 6/10 episodes completed
Grading progress: 7/10 episodes completed
Grading progress: 8/10 episodes completed
Grading progress: 9/10 episodes completed
Grading progress: 10/10 episodes completed
Student 20260000 final project score: -300/1000 (avg -30.00, completed 5/10, turns 56)
Amber Badger: 0/300 across variations [9, 10, 11]
Calm Badger: -200/300 across variations [9, 10, 11]
Clever Badger: -100/200 across variations [6, 7]
Brisk Badger: 0/200 across variations [6, 7]
Student 20260000 final project score: -300/1000 (avg -30.00, completed 5/10, turns 56)
Amber Badger: 0/300 across variations [9, 10, 11]
Calm Badger: -200/300 across variations [9, 10, 11]
Clever Badger: -100/200 across variations [6, 7]
Brisk Badger: 0/200 across variations [6, 7]
score: -300/1